# Training PPO - Coordinación de Agentes para Mitigación de Slippage (IPSA)

**Sprint 4 - Tarea 3.1.1/3.1.2/3.1.3/3.1.4/3.1.5**

Responsable: Paolo Sepúlveda (PS)

Fecha: Septiembre 2026

## 1. Setup & Dependencias

In [ ]:
# Instalar dependencias (solo primera vez)
!pip install gymnasium torch numpy scipy pandas matplotlib seaborn scikit-learn tqdm

In [ ]:
# Clonar repo si es necesario
import os
import sys

if not os.path.exists('/content/slippage'):
    !git clone https://github.com/Vatelfil/slippage.git /content/slippage

sys.path.insert(0, '/content/slippage')
os.chdir('/content/slippage')
print("✓ Repositorio listo")

In [ ]:
# Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal, Categorical
import gymnasium as gym
from collections import deque
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Imports del proyecto
from src.envs.spaces import MaestroSpace, EjecutorSpace, MaestroActionSpace, EjecutorActionSpace
from src.envs.maestro_env import MaestroEnv
from src.envs.ejecutor_env import EjecutorEnv

print("✓ Dependencias importadas")

## 2. Configuración de Hiperparámetros

In [ ]:
# Hiperparámetros PPO
config = {
    # Arquitectura
    'hidden_dim': 128,
    'num_layers': 2,
    'activation': 'relu',
    
    # Training
    'learning_rate': 3e-4,
    'gamma': 0.99,  # descuento
    'gae_lambda': 0.95,  # GAE
    'clip_ratio': 0.2,  # PPO clip
    'entropy_coef': 0.01,
    'value_coef': 0.5,
    
    # Rollout
    'max_steps': 30,  # pasos por episodio (30 min simulados)
    'num_envs': 4,  # ambientes paralelos
    'rollout_steps': 2048,  # samples antes de update
    'epochs_per_update': 10,
    
    # Training loop
    'total_timesteps': 1_000_000,
    'eval_interval': 50_000,
    'checkpoint_interval': 100_000,
    
    # Data
    'batch_size': 64,
    'seed': 42,
}

print(json.dumps(config, indent=2))

## 3. Redes Actor-Critic (Base)

In [ ]:
class ActorCritic(nn.Module):
    """Red Actor-Critic para PPO.
    
    Actor: π(a|s) - política estocástica
    Critic: V(s) - value function
    """
    
    def __init__(self, obs_dim, action_dim, hidden_dim=128, discrete=True):
        super().__init__()
        self.discrete = discrete
        
        # Shared backbone
        self.backbone = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Actor head
        if discrete:
            self.actor = nn.Linear(hidden_dim, action_dim)
        else:
            self.actor_mean = nn.Linear(hidden_dim, action_dim)
            self.actor_logstd = nn.Parameter(torch.zeros(1, action_dim))
        
        # Critic head
        self.critic = nn.Linear(hidden_dim, 1)
    
    def forward(self, obs):
        x = self.backbone(obs)
        value = self.critic(x)
        return x, value
    
    def get_action_and_value(self, obs, action=None):
        x, value = self.forward(obs)
        
        if self.discrete:
            logits = self.actor(x)
            probs = torch.softmax(logits, dim=-1)
            dist = Categorical(probs)
        else:
            mean = self.actor_mean(x)
            std = torch.exp(self.actor_logstd)
            dist = Normal(mean, std)
        
        if action is None:
            action = dist.sample()
        
        return action, dist.log_prob(action), dist.entropy(), value

print("✓ Arquitectura Actor-Critic definida")

## 4. Inicializar Ambientes

In [ ]:
# Maestro
maestro_env = MaestroEnv(max_steps=config['max_steps'])
maestro_obs_dim = maestro_env.observation_space.shape[0]
maestro_action_dim = 1  # continuo

print(f"Maestro:")
print(f"  Obs space: {maestro_obs_dim}D")
print(f"  Action space: {maestro_action_dim}D (continuo)")

# Ejecutores
ejecutor_envs = [EjecutorEnv(executor_id=i, max_steps=config['max_steps']) for i in range(3)]
ejecutor_obs_dim = ejecutor_envs[0].observation_space.shape[0]
ejecutor_action_dim = 3  # discreto: (order_type, volume, price_level)

print(f"\nEjecutores (x3):")
print(f"  Obs space: {ejecutor_obs_dim}D")
print(f"  Action space: {ejecutor_action_dim}D (discreto)")

## 5. Inicializar Redes (usar las de Mauricio cuando estén listas)

In [ ]:
# TODO: Reemplazar con redes de Mauricio (src/models/actor_critic.py)
# Por ahora usamos implementación base

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Maestro
maestro_actor_critic = ActorCritic(
    obs_dim=maestro_obs_dim,
    action_dim=maestro_action_dim,
    hidden_dim=config['hidden_dim'],
    discrete=False
).to(device)

maestro_optimizer = optim.Adam(maestro_actor_critic.parameters(), lr=config['learning_rate'])

# Ejecutores
ejecutor_actor_critics = [
    ActorCritic(
        obs_dim=ejecutor_obs_dim,
        action_dim=ejecutor_action_dim,
        hidden_dim=config['hidden_dim'],
        discrete=True
    ).to(device)
    for _ in range(3)
]

ejecutor_optimizers = [optim.Adam(ac.parameters(), lr=config['learning_rate']) for ac in ejecutor_actor_critics]

print("✓ Redes inicializadas")

## 6. Buffer de Rollout (Experiencias)

In [ ]:
class RolloutBuffer:
    """Buffer para almacenar experiencias del rollout."""
    
    def __init__(self, capacity):
        self.capacity = capacity
        self.clear()
    
    def clear(self):
        self.obs = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
        self.dones = []
        self.returns = []
        self.advantages = []
    
    def add(self, obs, action, log_prob, reward, value, done):
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)
    
    def compute_returns_advantages(self, gamma, gae_lambda):
        """Compute GAE advantages and returns."""
        self.returns = []
        self.advantages = []
        
        gae = 0
        for t in reversed(range(len(self.rewards))):
            if t == len(self.rewards) - 1:
                next_value = 0
            else:
                next_value = self.values[t + 1]
            
            delta = self.rewards[t] + gamma * next_value - self.values[t]
            gae = delta + gamma * gae_lambda * gae
            
            self.advantages.insert(0, gae)
            self.returns.insert(0, gae + self.values[t])
    
    def get_batches(self, batch_size):
        """Yield minibatches."""
        n = len(self.obs)
        indices = np.random.permutation(n)
        
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch_indices = indices[start:end]
            
            yield {
                'obs': torch.FloatTensor(np.array([self.obs[i] for i in batch_indices])).to(device),
                'actions': torch.LongTensor([self.actions[i] for i in batch_indices]).to(device),
                'log_probs': torch.FloatTensor([self.log_probs[i] for i in batch_indices]).to(device),
                'returns': torch.FloatTensor([self.returns[i] for i in batch_indices]).to(device),
                'advantages': torch.FloatTensor([self.advantages[i] for i in batch_indices]).to(device),
            }

print("✓ RolloutBuffer definido")

## 7. Training Loop PPO (Skeleton)

In [ ]:
def ppo_update(actor_critic, optimizer, batch, clip_ratio, entropy_coef, value_coef):
    """Actualiza la red con PPO loss."""
    
    # Forward pass
    # TODO: Implementar PPO loss completo
    # Por ahora es skeleton
    
    return {
        'policy_loss': 0.0,
        'value_loss': 0.0,
        'entropy': 0.0,
    }

# Tracking
train_stats = {
    'maestro_return': deque(maxlen=100),
    'ejecutor_return': [deque(maxlen=100) for _ in range(3)],
    'losses': [],
}

print("✓ Training loop skeleton definido")

## 8. Checkpointing & Logging

In [ ]:
import os
from datetime import datetime

# Crear directorio de checkpoints
checkpoint_dir = Path('checkpoints') / f"ppo_sprint4_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print(f"Checkpoint dir: {checkpoint_dir}")

def save_checkpoint(step):
    """Guardar estado del training."""
    checkpoint = {
        'step': step,
        'maestro_state_dict': maestro_actor_critic.state_dict(),
        'maestro_optimizer': maestro_optimizer.state_dict(),
        'ejecutor_state_dicts': [ac.state_dict() for ac in ejecutor_actor_critics],
        'ejecutor_optimizers': [opt.state_dict() for opt in ejecutor_optimizers],
        'train_stats': dict(train_stats),
    }
    torch.save(checkpoint, checkpoint_dir / f'checkpoint_{step}.pt')
    print(f"✓ Checkpoint guardado: {step}")

print("✓ Checkpointing definido")

## 9. Validación (Evaluation)

In [ ]:
def evaluate(n_episodes=5):
    """Evaluar política sin exploración."""
    
    maestro_returns = []
    ejecutor_returns = [[] for _ in range(3)]
    
    for ep in range(n_episodes):
        # Reset
        maestro_obs, _ = maestro_env.reset()
        ejecutor_obss = [env.reset()[0] for env in ejecutor_envs]
        
        maestro_ep_return = 0
        ejecutor_ep_returns = [0] * 3
        done = False
        
        while not done:
            # Maestro step
            # TODO: step maestro
            
            # Ejecutores step
            for i in range(3):
                # TODO: step ejecutor i
                pass
        
        maestro_returns.append(maestro_ep_return)
        for i in range(3):
            ejecutor_returns[i].append(ejecutor_ep_returns[i])
    
    return {
        'maestro_mean': np.mean(maestro_returns),
        'ejecutor_means': [np.mean(r) for r in ejecutor_returns],
    }

print("✓ Evaluation function definida")

## 10. Análisis de Convergencia

In [ ]:
def plot_convergence():
    """Plotear métricas de convergencia."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # TODO: Plotear train_stats
    
    plt.tight_layout()
    plt.savefig(checkpoint_dir / 'convergence.png', dpi=100)
    plt.show()

print("✓ Análisis de convergencia definido")

## 11. MAIN: Training Loop (Cuando esté listo)

In [ ]:
# TODO: Implementar training loop completo
# 
# Estructura:
# 1. Rollout: Ejecutar episodios y recolectar experiencias
# 2. Compute advantages: GAE
# 3. PPO updates: clip_ratio, entropy, value losses
# 4. Logging: loss, returns, etc
# 5. Checkpointing: cada eval_interval
# 6. Evaluación: cada checkpoint
#
# Dependencias:
# - Redes de Mauricio (src/models/actor_critic.py)
# - Test ABIDES de Mauricio (src/envs/test_abides.py)
# - Parámetros Poisson reales (cuando Benjamín entregue data/processed/)

print("""\n⏳ TRAINING LOOP SKELETON LISTO\n
Proximos pasos:
1. Esperar redes de Mauricio (2.1.1/2.1.2)
2. Integrar ABIDES
3. Implementar PPO loss completo
4. Ejecutar training con hyperparameters afinados
5. Guardar checkpoints y logs""")

---

**Estado**: Skeleton lista

**Bloqueadores**: Redes de Mauricio, ABIDES integration

**Próxima reunión**: Coordinar con Mauricio/Benjamín para integración